# Layer 5 — Model Deployment: Containerized Inference Endpoints & Traffic Routing

**Architectural Layer:** Model Deployment  
**Toolchain:** BentoML · vLLM · Docker · Kubernetes  
**Objective:** Package a model into a production container, generate Kubernetes manifests for autoscaling, and benchmark latency.

---

## 🧠 What is Model Deployment and Why Is It the Hardest Part?

**Real-world scenario:**  
Your data scientist trained an amazing model. It gets 95% accuracy. Everyone is excited. Now... how do you actually make it available to millions of users? This is deployment — and it's where most ML projects fail.

### 🤔 Why Can't I Just Run `python predict.py` on a Server?

| Challenge | Why It's a Problem |
|-----------|-------------------|
| **Scalability** | 1 Python process handles ~10 requests/sec. You need 10,000/sec. |
| **Reliability** | If the process crashes, all users get errors. |
| **Latency** | Loading the model takes 30 seconds. Each request can't wait 30 seconds. |
| **GPU utilization** | A $10,000 GPU sits idle 90% of the time without batching. |
| **Versioning** | Which model version is running? How do you rollback? |
| **Security** | Is the container image free of vulnerabilities? |

### The Deployment Stack

```
User Request → Load Balancer → Kubernetes → Pod (Container)
                                                  │
                                           BentoML Service
                                                  │
                                           vLLM Engine
                                                  │
                                           GPU (inference)
```

Each layer solves a different problem:
- **BentoML**: Packages the model as an API with proper input validation
- **vLLM**: Optimizes LLM inference (batching, KV-cache, PagedAttention)
- **Docker**: Creates a portable, reproducible container
- **Kubernetes**: Manages scaling, health checks, and traffic routing

### ⚖️ Trade-offs: Self-Hosted vs. Managed Inference

| Approach | Pros | Cons | Cost |
|----------|------|------|------|
| **Self-hosted (this notebook)** | Full control, customization | You manage infrastructure | Medium-high |
| **Managed (SageMaker, Vertex AI)** | Easy setup, auto-scaling | Vendor lock-in, less customization | High |
| **Serverless (Lambda, Cloud Run)** | Pay per request, no idle cost | Cold starts, size limits | Low-medium |

**🤔 When to use each:**
- **Self-hosted**: Large LLMs, custom inference logic, cost-sensitive at scale
- **Managed**: Quick deployment, small team, don't want to manage K8s
- **Serverless**: Lightweight models (< 500MB), bursty traffic patterns

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Wrapping a PyTorch model in Flask is a recipe for disaster (GIL locks, poor batching). Seniors use specialized model servers (vLLM, BentoML) that handle dynamic batching, multi-threading, and GPU memory natively.

**Mental Model:** A standard web server handles requests 1-by-1. A GPU model server waits a few milliseconds to gather 10 incoming requests, bundles them into a matrix block (batching), and pushes them to the GPU simultaneously.

**Common Junior Pitfall:** Not pre-allocating GPU memory. Loading a model lazily on the first request causes a massive 10-second latency spike for the first user. Seniors load weights into VRAM on startup.

---


## 📚 Key Concepts Explained

### What is vLLM and Why Not Just Use Transformers?

HuggingFace Transformers is great for research but slow for production. vLLM is 5-24x faster because of:

| Feature | Transformers | vLLM |
|---------|-------------|------|
| **Continuous batching** | No (waits for all to finish) | Yes (new requests join mid-batch) |
| **PagedAttention** | No (wastes GPU memory) | Yes (efficient memory management) |
| **KV-cache optimization** | Basic | Advanced (reuses computed values) |
| **Throughput (tokens/sec)** | ~100 | ~500-2000+ |

### What is TTFT and Why Does It Matter?

- **TTFT (Time to First Token)**: How long until the user sees the first word of the response
- **E2E Latency**: Total time for the complete response

**Why TTFT matters:** Users perceive response time based on TTFT, not E2E. A chatbot that starts typing in 200ms feels fast, even if the full response takes 3 seconds. A chatbot that shows nothing for 3 seconds feels broken.

### What is a Kubernetes Deployment?

**Analogy:** Kubernetes is like a restaurant manager. It ensures:
- Enough chefs (pods/containers) are working during rush hour
- If a chef gets sick (pod crashes), a replacement is assigned immediately
- When it's slow (low traffic), extra chefs go home (scale down)
- New menu items (model versions) are introduced without closing the restaurant (zero-downtime deployment)

---

In [ ]:
# Cell 1 — Install dependencies
!pip install -q bentoml vllm aiohttp matplotlib numpy pyyaml

## 1 · BentoML Service Definition

### 🤔 What is a BentoML Service?

A BentoML Service is a Python class that wraps your model and exposes it as an API endpoint. It handles:
- Input validation (reject malformed requests)
- Model loading (load once, serve many)
- Response formatting (consistent JSON output)
- Health checks ("is the service alive?")

### 🤔 Why `/generate` AND `/stream`?

| Endpoint | Returns | Use Case |
|----------|---------|----------|
| `/generate` | Full response at once | Backend API calls, batch processing |
| `/stream` | Token by token (SSE) | Chatbots, real-time UI |

**Real-world scenario:** ChatGPT uses streaming. When you type a question, tokens appear one by one. This is the `/stream` endpoint pattern. Behind the scenes, each token is sent as a Server-Sent Event (SSE).

In [ ]:
# Cell 2 — Define BentoML service with vLLM backend
#
# WHAT IS HAPPENING?
# We create a Python class that:
# 1. Loads the vLLM engine when the service starts (__init__)
# 2. Exposes a /generate endpoint for full responses
# 3. Exposes a /stream endpoint for token-by-token streaming
#
# KEY DETAILS:
# - tensor_parallel_size=1: use 1 GPU (set to 2/4/8 for multi-GPU)
# - max_tokens: prevent runaway generation (cost + latency control)
# - temperature: controls randomness (0 = deterministic, 1 = creative)
#
# 🤔 Why write this to a file instead of running it here?
# BentoML services run as standalone processes, not in notebooks.
# We write the code to a file that BentoML can load and deploy.

service_code = '''
import bentoml
from vllm import LLM, SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
import asyncio
import uuid

@bentoml.service(
    name="llm-inference-service",
    resources={"gpu": 1, "memory": "16Gi"},
    traffic={"timeout": 120},  # max 120 seconds per request
)
class LLMService:
    def __init__(self):
        """Initialize the vLLM engine. This runs ONCE when the service starts.
        
        The model is loaded into GPU memory here. All subsequent requests
        reuse the loaded model — no reloading per request.
        """
        engine_args = AsyncEngineArgs(
            model="microsoft/phi-2",       # the model to serve
            tensor_parallel_size=1,         # GPUs to split across
            max_model_len=2048,             # maximum context window
            gpu_memory_utilization=0.85,    # use 85% of GPU memory
            enforce_eager=False,            # enable CUDA graphs for speed
        )
        self.engine = AsyncLLMEngine.from_engine_args(engine_args)

    @bentoml.api()
    async def generate(self, prompt: str, max_tokens: int = 512,
                       temperature: float = 0.7) -> dict:
        """Generate a complete response and return it all at once.
        
        Use this for:
        - Backend API calls
        - Batch processing
        - When you need the full response before processing
        """
        sampling_params = SamplingParams(
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=0.95,
        )
        request_id = str(uuid.uuid4())
        results_generator = self.engine.generate(prompt, sampling_params, request_id)
        final_output = None
        async for output in results_generator:
            final_output = output
        return {
            "text": final_output.outputs[0].text,
            "tokens_generated": len(final_output.outputs[0].token_ids),
            "finish_reason": final_output.outputs[0].finish_reason,
        }

    @bentoml.api()
    async def stream(self, prompt: str, max_tokens: int = 512,
                     temperature: float = 0.7):
        """Stream tokens one by one as they are generated.
        
        Use this for:
        - Chatbot UIs (show tokens as they appear)
        - Real-time applications
        - When perceived latency matters more than total latency
        """
        sampling_params = SamplingParams(
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=0.95,
        )
        request_id = str(uuid.uuid4())
        async for output in self.engine.generate(prompt, sampling_params, request_id):
            yield output.outputs[0].text
'''

with open("service.py", "w") as f:
    f.write(service_code)

print("✅ BentoML service written to service.py")
print("   Endpoints: /generate (full response), /stream (token-by-token)")

---

## 2 · Container Build & Security Scanning

### 🤔 What is a Dockerfile?

A Dockerfile is a recipe for building a container image. It specifies:
1. Base image (the starting point, e.g., NVIDIA CUDA runtime)
2. Dependencies to install
3. Files to copy into the image
4. Command to run when the container starts

### 🤔 Why Scan Containers for Vulnerabilities?

Container images contain hundreds of packages. Any one of them might have a known security vulnerability (CVE). If you deploy an image with a critical vulnerability, attackers can exploit it.

**Real-world scenario:**  
A company deployed an ML service with an outdated OpenSSL version in the container. An attacker exploited the vulnerability to access the GPU cluster and mine cryptocurrency, costing thousands in compute bills.

**Rule: Never deploy an image with CRITICAL vulnerabilities. HIGH vulnerabilities should be reviewed.**

In [ ]:
# Cell 3 — Generate BentoML build configuration
#
# bentofile.yaml tells BentoML HOW to build the container.
# It's like package.json for Node.js — lists dependencies and configuration.
#
# KEY SECTIONS:
# - python.packages: Python libraries to install
# - docker.base_image: start from NVIDIA's CUDA-enabled image
#   (without this, the container can't use GPUs)
# - docker.env: environment variables inside the container

import yaml

bentofile = {
    "service": "service:LLMService",
    "labels": {"team": "ml-platform", "model": "phi-2", "framework": "vllm"},
    "include": ["*.py"],
    "python": {
        "packages": ["vllm>=0.5.0", "transformers>=4.40.0", "torch>=2.3.0", "accelerate"],
    },
    "docker": {
        "base_image": "nvidia/cuda:12.4.0-runtime-ubuntu22.04",
        "env": {
            "CUDA_VISIBLE_DEVICES": "0",
            "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
            "HF_HOME": "/home/bentoml/.cache/huggingface",
        },
    },
}

with open("bentofile.yaml", "w") as f:
    yaml.dump(bentofile, f, default_flow_style=False)

print("📦 bentofile.yaml generated:")
print(yaml.dump(bentofile, default_flow_style=False))

In [ ]:
# Cell 4 — Simulate container build and vulnerability scan
#
# In production, this is automated in your CI/CD pipeline (Notebook 04).
# The flow: build image → scan for vulnerabilities → push to registry
# If scan finds CRITICAL issues → build FAILS → image is NOT pushed.

print("🐳 Simulating container build...")
print("   Step 1/5: FROM nvidia/cuda:12.4.0-runtime-ubuntu22.04")
print("   Step 2/5: RUN pip install vllm transformers torch")
print("   Step 3/5: COPY service.py /home/bentoml/bento/")
print("   Step 4/5: Setting environment variables")
print("   Step 5/5: CMD [\"bentoml\", \"serve\", \"service:LLMService\"]")
print("   ✅ Image built: registry.company.com/llm-service:v3-abc123")

# Simulated Trivy scan report
scan_report = {
    "image": "llm-service:v3-abc123",
    "scan_time": "2026-02-28T18:00:00Z",
    "vulnerabilities": {
        "CRITICAL": 0,
        "HIGH": 2,
        "MEDIUM": 8,
        "LOW": 15,
    },
    "verdict": "PASS (0 critical)",
}
print(f"\n🔍 Vulnerability Scan Results:")
for severity, count in scan_report["vulnerabilities"].items():
    icon = "🔴" if severity == "CRITICAL" and count > 0 else "🟢"
    print(f"   {icon} {severity}: {count}")
print(f"   Verdict: {scan_report['verdict']}")

---

## 3 · Kubernetes Manifests

### 🤔 What Are Kubernetes Manifests?

Manifests are YAML files that tell Kubernetes: "I want THIS to exist." You describe the DESIRED state, and Kubernetes makes it happen.

| Manifest Type | What It Does | Analogy |
|--------------|-------------|--------|
| **Deployment** | "Run 3 copies of this container" | "Hire 3 chefs" |
| **Service** | "Route traffic to these containers" | "Set up the order window" |
| **HPA** | "Add/remove containers based on load" | "Call in backup chefs when it's busy" |
| **Ingress** | "Make this accessible from the internet" | "Put a sign on the street" |

### 🤔 What is Horizontal Pod Autoscaler (HPA)?

HPA automatically scales the number of pods (containers) based on metrics:

```
Low traffic  → 2 pods  (save money)
Normal traffic → 4 pods
Peak traffic  → 8 pods  (handles load)
```

**⚖️ Trade-off: Scaling Speed vs. Cost**

| Setting | Pros | Cons |
|---------|------|------|
| Scale up aggressively | Handles traffic spikes quickly | Wastes money during false alarms |
| Scale up conservatively | Cost-efficient | May drop requests during sudden spikes |
| Never scale down below 2 | Always available | Paying for idle capacity |

In [ ]:
# Cell 5 — Generate Kubernetes Deployment manifest
#
# KEY DETAILS:
# - resources.limits: maximum resources a pod can use
#   nvidia.com/gpu: 1 → each pod gets exactly 1 GPU
# - readinessProbe: K8s checks /healthz to know if the pod is ready
#   If it's not ready, K8s doesn't send traffic to it
#   initialDelaySeconds: 30 → wait 30s before first check (model loading time)
# - replicas: 2 → start with 2 pods for redundancy
#   If one pod crashes, the other keeps serving

deployment_manifest = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": "llm-inference", "namespace": "ml-serving",
                 "labels": {"app": "llm-inference", "version": "v3"}},
    "spec": {
        "replicas": 2,
        "selector": {"matchLabels": {"app": "llm-inference"}},
        "template": {
            "metadata": {"labels": {"app": "llm-inference", "version": "v3"}},
            "spec": {
                "containers": [{
                    "name": "llm-service",
                    "image": "registry.company.com/llm-service:v3-abc123",
                    "ports": [{"containerPort": 3000, "name": "http"}],
                    "resources": {
                        "requests": {"cpu": "4", "memory": "16Gi", "nvidia.com/gpu": "1"},
                        "limits": {"cpu": "8", "memory": "32Gi", "nvidia.com/gpu": "1"},
                    },
                    "readinessProbe": {
                        "httpGet": {"path": "/healthz", "port": 3000},
                        "initialDelaySeconds": 30,
                        "periodSeconds": 10,
                    },
                }],
                "tolerations": [{"key": "nvidia.com/gpu", "operator": "Exists", "effect": "NoSchedule"}],
            },
        },
    },
}

with open("k8s_deployment.yaml", "w") as f:
    yaml.dump(deployment_manifest, f, default_flow_style=False)
print("✅ Deployment manifest generated → k8s_deployment.yaml")

In [ ]:
# Cell 6 — Generate HPA, Service, and Ingress manifests

hpa_manifest = {
    "apiVersion": "autoscaling/v2",
    "kind": "HorizontalPodAutoscaler",
    "metadata": {"name": "llm-inference-hpa", "namespace": "ml-serving"},
    "spec": {
        "scaleTargetRef": {"apiVersion": "apps/v1", "kind": "Deployment", "name": "llm-inference"},
        "minReplicas": 2,   # never go below 2 (redundancy)
        "maxReplicas": 8,   # never exceed 8 (cost control)
        "metrics": [
            {"type": "Resource", "resource": {"name": "cpu", "target": {"type": "Utilization", "averageUtilization": 70}}},
        ],
    },
}

service_manifest = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": "llm-inference-svc", "namespace": "ml-serving"},
    "spec": {
        "selector": {"app": "llm-inference"},
        "ports": [{"port": 80, "targetPort": 3000, "protocol": "TCP"}],
        "type": "ClusterIP",
    },
}

ingress_manifest = {
    "apiVersion": "networking.k8s.io/v1",
    "kind": "Ingress",
    "metadata": {"name": "llm-inference-ingress", "namespace": "ml-serving",
                 "annotations": {"nginx.ingress.kubernetes.io/ssl-redirect": "true"}},
    "spec": {
        "rules": [{
            "host": "llm-api.company.com",
            "http": {"paths": [{"path": "/", "pathType": "Prefix",
                    "backend": {"service": {"name": "llm-inference-svc", "port": {"number": 80}}}}]},
        }],
    },
}

for name, manifest in [("hpa", hpa_manifest), ("service", service_manifest), ("ingress", ingress_manifest)]:
    with open(f"k8s_{name}.yaml", "w") as f:
        yaml.dump(manifest, f, default_flow_style=False)
    print(f"✅ {name.upper()} manifest → k8s_{name}.yaml")

print(f"\n📋 Scaling: {hpa_manifest['spec']['minReplicas']}-{hpa_manifest['spec']['maxReplicas']} replicas")

---

## 4 · Load Testing & Benchmarking

### 🤔 Why Load Test Before Going Live?

**Real-world scenario:**  
A startup launched their AI chatbot on launch day. Within 2 hours, their service crashed under load. They hadn't tested with more than 10 concurrent users. They had 10,000.

Load testing answers:
- How many requests per second can my service handle?
- What's the latency at 50%, 95%, and 99% of requests?
- At what load does the service degrade or crash?

### 🤔 What Metrics Should You Track?

| Metric | What It Measures | Acceptable | Concerning |
|--------|-----------------|------------|------------|
| **TTFT** | Time to first token | < 200ms | > 1000ms |
| **E2E Latency** | Total response time | < 3s (128 tokens) | > 10s |
| **Throughput (TPS)** | Tokens per second | > 100 TPS | < 30 TPS |
| **Error Rate** | % of failed requests | < 0.1% | > 1% |
| **GPU Utilization** | How busy the GPU is | 70-85% | > 95% or < 30% |

In [ ]:
# Cell 7 — Simulate load testing
#
# In production, you'd use real tools like:
# - Locust (Python-based load testing)
# - k6 (JavaScript-based, very popular)
# - vegeta (Go-based, simple CLI)
#
# Here we simulate results to show what you'd analyze.

import numpy as np

np.random.seed(42)
NUM_REQUESTS = 1000

# Simulate realistic latency distributions
ttft_ms = np.random.lognormal(mean=4.5, sigma=0.5, size=NUM_REQUESTS)    # 50-200ms typical
e2e_ms = np.random.lognormal(mean=6.5, sigma=0.6, size=NUM_REQUESTS)      # 500-2000ms typical
tokens_generated = np.random.randint(50, 512, NUM_REQUESTS)
throughput = tokens_generated / (e2e_ms / 1000)  # tokens per second

print(f"📊 Load Test Results ({NUM_REQUESTS} requests):")
print(f"\n   TTFT (Time to First Token):")
print(f"     p50: {np.percentile(ttft_ms, 50):.0f} ms")
print(f"     p95: {np.percentile(ttft_ms, 95):.0f} ms")
print(f"     p99: {np.percentile(ttft_ms, 99):.0f} ms")
print(f"\n   E2E Latency:")
print(f"     p50: {np.percentile(e2e_ms, 50):.0f} ms")
print(f"     p95: {np.percentile(e2e_ms, 95):.0f} ms")
print(f"     p99: {np.percentile(e2e_ms, 99):.0f} ms")
print(f"\n   Throughput: {np.median(throughput):.0f} tokens/sec (median)")

In [ ]:
# Cell 8 — Visualize latency distributions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(ttft_ms, bins=50, color="#2196F3", alpha=0.8, edgecolor="white")
axes[0].axvline(np.percentile(ttft_ms, 95), color="red", linestyle="--", label=f"p95: {np.percentile(ttft_ms, 95):.0f}ms")
axes[0].set_title("TTFT Distribution", fontweight="bold")
axes[0].set_xlabel("Milliseconds")
axes[0].legend()

axes[1].hist(e2e_ms, bins=50, color="#FF5722", alpha=0.8, edgecolor="white")
axes[1].axvline(np.percentile(e2e_ms, 95), color="red", linestyle="--", label=f"p95: {np.percentile(e2e_ms, 95):.0f}ms")
axes[1].set_title("E2E Latency Distribution", fontweight="bold")
axes[1].set_xlabel("Milliseconds")
axes[1].legend()

axes[2].hist(throughput, bins=50, color="#4CAF50", alpha=0.8, edgecolor="white")
axes[2].set_title("Throughput Distribution", fontweight="bold")
axes[2].set_xlabel("Tokens/sec")

plt.tight_layout()
plt.savefig("benchmark_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("📈 Benchmark visualizations saved")

In [ ]:
# Cell 9 — Deployment readiness assessment
#
# Before going live, verify that ALL criteria are met.
# This is like a pilot's pre-flight checklist.

checks = [
    ("Container built & pushed", True),
    ("Vulnerability scan passed (0 CRITICAL)", True),
    ("K8s manifests generated", True),
    ("HPA configured (2-8 replicas)", True),
    ("TTFT p95 < 500ms", np.percentile(ttft_ms, 95) < 500),
    ("E2E p95 < 5000ms", np.percentile(e2e_ms, 95) < 5000),
    ("Error rate < 1%", True),
    ("Health check endpoint configured", True),
    ("SSL/TLS configured", True),
]

print("🚀 Deployment Readiness Checklist:")
all_passed = True
for check, passed in checks:
    icon = "✅" if passed else "❌"
    print(f"   {icon} {check}")
    if not passed: all_passed = False

print(f"\n{'🟢 READY FOR DEPLOYMENT' if all_passed else '🔴 NOT READY — fix failed checks'}")

---

## 5 · FastAPI Serving with Pydantic Validation

### 🤔 Why FastAPI for ML Serving?

| Framework | Speed | Validation | Docs | Best For |
|-----------|-------|-----------|------|----------|
| **FastAPI** | Very fast (async) | Built-in (Pydantic) | Auto-generated | Custom ML APIs |
| **Flask** | Moderate | Manual | Manual | Simple prototypes |
| **BentoML** | Fast | Built-in | Built-in | Standardized model serving |

**Pydantic** ensures that every request has the correct data types BEFORE reaching your model. Without it, invalid inputs cause cryptic crashes.

In [ ]:
# Cell — FastAPI model serving with Pydantic input validation
#
# This is a production-ready API structure.
# Pydantic validates inputs, FastAPI handles routing, and we track latency.

fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel, Field
import pickle, time, numpy as np

app = FastAPI(title="ML Inference API", version="1.0")

# Pydantic ensures type safety — rejects invalid requests automatically
class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=100, description="Applicant age")
    income: float = Field(gt=0, description="Annual income in USD")
    credit_score: int = Field(ge=300, le=850, description="Credit score")
    employment_years: int = Field(ge=0, le=50)

class PredictionResponse(BaseModel):
    prediction: int
    probability: float
    latency_ms: float
    model_version: str

# Load model at startup (not per-request!)
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict", response_model=PredictionResponse)
def predict(req: PredictionRequest):
    start = time.perf_counter()
    features = np.array([[req.age, req.income, req.credit_score, req.employment_years]])
    pred = model.predict(features)[0]
    proba = model.predict_proba(features)[0].max()
    latency = (time.perf_counter() - start) * 1000
    return PredictionResponse(prediction=int(pred), probability=float(proba),
                              latency_ms=round(latency, 2), model_version="v1.0")
'''

with open('fastapi_service.py', 'w') as f:
    f.write(fastapi_code)
print("📄 FastAPI service written to fastapi_service.py")
print("   Run with: uvicorn fastapi_service:app --host 0.0.0.0 --port 8000")
print("   Docs at: http://localhost:8000/docs (auto-generated!)")
print(fastapi_code)

---
## Knowledge Check

### Q1: Cold Start
Your K8s HPA scales from 2 to 4 pods during a traffic spike. The new pods take 45 seconds to load the model. What happens to requests during those 45 seconds?

<details><summary>Answer</summary>

The `readinessProbe` prevents K8s from sending traffic to new pods until they're ready. So during those 45s, only the original 2 pods handle traffic (possibly with higher latency). Requests are NOT routed to the loading pods. This is why `initialDelaySeconds` matters.
</details>

### Q2: GPU Memory
You set `gpu_memory_utilization=0.95` in vLLM. Why might this cause problems?

<details><summary>Answer</summary>

At 95%, there's almost no headroom for KV-cache growth during inference. When concurrent requests spike, the KV-cache may exceed available memory, causing OOM errors and pod crashes. 0.85 is safer -- it leaves 15% for KV-cache and runtime overhead.
</details>

### Q3: Vulnerability Scan
Your container scan shows 0 CRITICAL but 12 HIGH vulnerabilities. Should you deploy?

<details><summary>Answer</summary>

It depends. Review each HIGH vulnerability: are they in packages your service actually uses? Are there known exploits? If they're in unused packages, you can deploy with a remediation timeline. If they're in actively-used libraries (e.g., OpenSSL), fix before deploying.
</details>


---
## Practical Practice

### Exercise 1: Write a readiness check endpoint that verifies the model is loaded AND can process a test input.
### Exercise 2: Calculate how many A100 GPUs (80GB) you need to serve a 70B model at 4-bit with 100 concurrent users.


In [ ]:
# ===== SOLUTIONS =====
print('Ex 1: Readiness Check')
print('''@app.get("/healthz")
async def readiness():
    if model is None:
        return JSONResponse({"ready": False}, status_code=503)
    try:
        _ = model.generate("test", max_tokens=1)  # Smoke test
        return {"ready": True, "model": "loaded"}
    except Exception as e:
        return JSONResponse({"ready": False, "error": str(e)}, status_code=503)''')

print('\nEx 2: GPU Calculation')
model_size_gb = 70 * 4 / 8  # 70B params at 4-bit = 35GB
kv_per_user = 0.5  # ~0.5GB KV-cache per concurrent user
total_kv = 100 * kv_per_user  # 100 users = 50GB
total_need = model_size_gb + total_kv
gpus_needed = -(-int(total_need) // 80)  # Ceiling division by 80GB
print(f'  Model: {model_size_gb}GB + KV-cache: {total_kv}GB = {total_need}GB')
print(f'  A100 80GB GPUs needed: {gpus_needed}')
print(f'  Use tensor_parallel_size={gpus_needed} in vLLM config')


---

## 🎯 Summary & Key Takeaways

| Step | Action | Tool | Why |
|------|--------|------|-----|
| 1 | BentoML service with generate + stream | BentoML + vLLM | High-throughput inference with proper API |
| 2 | Container build with vulnerability scanning | Docker + Trivy | Secure, reproducible deployments |
| 3 | K8s manifests (Deployment, HPA, Service, Ingress) | Kubernetes | Scalable, self-healing infrastructure |
| 4 | Load testing and readiness assessment | Custom benchmarks | Prevent production outages |

### 🧠 Key Questions

1. **"How many GPUs do I need?"** → Run load tests at expected peak traffic. Add 50% buffer.
2. **"What if the model is too slow?"** → Use vLLM (batching), quantize the model, or use a smaller model.
3. **"How do I do zero-downtime updates?"** → K8s rolling updates: new pods start before old ones stop.

**Next -->** `07_monitoring_feedback.ipynb` -- Monitoring & Feedback Loops
